# Feature Selection — Selección de features

**Objetivo:** Seleccionar el set final de features para el modelo.

Estrategia:
1. Cargar resumen de tests estadísticos
2. Mantener solo features con evidencia (p < 0.05 Y efecto ≥ 0.1)
3. Verificar multicolinealidad (correlación > 0.9)
4. Guardar set final

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('Setup listo.')

---
## 2. Carga de datos

In [ ]:
df = pd.read_csv('../data/processed/application_train_preprocessed.csv')
tests = pd.read_csv('../data/processed/statistical_tests_summary.csv')

TARGET_COL = 'TARGET'
ID_COL = 'SK_ID_CURR'

print(f'Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Tests: {tests.shape[0]} features evaluadas')

---
## 3. Paso 1 — Filtrar por evidencia estadística

Mantenemos solo features con:
- p < 0.05 (significativas)
- efecto ≥ 0.1 (utilidad práctica)

In [ ]:
util_features = tests[tests['Util_practica'] == True]['Feature'].tolist()
no_util = tests[tests['Util_practica'] == False]['Feature'].tolist()

print(f'Features con evidencia: {len(util_features)}')
print(f'Features sin evidencia: {len(no_util)}')
print()
print('Features seleccionadas:')
for f in util_features:
    row = tests[tests['Feature'] == f].iloc[0]
    print(f'  {f:45s} {row["Tipo"]:12s} efecto={row["Effect_size"]:.3f}')

---
## 4. Paso 2 — Verificar multicolinealidad

Si dos features tienen correlación > 0.9, una es redundante.

In [ ]:
# Matriz de correlación de features seleccionadas
corr_matrix = df[util_features].corr().abs()

# Encontrar pares altamente correlacionados
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > 0.9:
            high_corr.append({
                'Feature_1': corr_matrix.columns[i],
                'Feature_2': corr_matrix.columns[j],
                'Correlacion': corr_matrix.iloc[i, j]
            })

high_corr_df = pd.DataFrame(high_corr).sort_values('Correlacion', ascending=False)

print(f'Pares con correlación > 0.9: {len(high_corr_df)}')
print()
high_corr_df

In [ ]:
# Eliminar una de cada par altamente correlacionado
# Estrategia: eliminar la que tiene menor correlación con TARGET
corr_with_target = df.drop(columns=[ID_COL]).corr(method='spearman')[TARGET_COL]

cols_to_remove = set()
for _, row in high_corr_df.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    if f1 in cols_to_remove or f2 in cols_to_remove:
        continue
    
    corr_f1 = abs(corr_with_target.get(f1, 0))
    corr_f2 = abs(corr_with_target.get(f2, 0))
    
    remove = f2 if corr_f1 >= corr_f2 else f1
    keep = f1 if corr_f1 >= corr_f2 else f2
    
    cols_to_remove.add(remove)
    print(f'  Eliminar: {remove:40s} (corr_target={corr_with_target.get(remove, 0):.4f})')
    print(f'  Mantener: {keep:40s} (corr_target={corr_with_target.get(keep, 0):.4f})')
    print()

final_features = [f for f in util_features if f not in cols_to_remove]

print(f'Después de multicolinealidad: {len(final_features)} features')

---
## 5. Resumen final

In [ ]:
print('RESUMEN DE SELECCIÓN')
print('=' * 50)
print(f'Features iniciales:      {df.shape[1] - 2}')
print(f'Después de tests:         {len(util_features)}')
print(f'Después de multicolineal: {len(final_features)}')
print()
print('Features finales:')
for i, f in enumerate(final_features, 1):
    row = tests[tests['Feature'] == f].iloc[0]
    print(f'  {i:2d}. {f:45s} {row["Tipo"]:12s} efecto={row["Effect_size"]:.3f}')

---
## 6. Guardar dataset final

In [ ]:
df_final = df[[ID_COL, TARGET_COL] + final_features]

import os
os.makedirs('../data/processed', exist_ok=True)

df_final.to_csv('../data/processed/application_train_selected.csv', index=False)

print(f'Dataset final: {df_final.shape[0]:,} filas x {df_final.shape[1]} columnas')
print(f'  Features: {len(final_features)}')
print(f'  Target: 1')
print(f'  ID: 1')

In [ ]:
# Guardar lista de features
with open('../data/processed/selected_features.txt', 'w') as f:
    for feat in final_features:
        f.write(feat + '\n')

print(f'Guardado: data/processed/selected_features.txt ({len(final_features)} features)')

---
## 7. Próximos pasos

En `08_modeling.ipynb`:
1. Cargar `application_train_selected.csv`
2. Train/test split (80/20, stratified)
3. Scaling (StandardScaler)
4. Entrenar modelos baseline
5. Evaluar con métricas para desbalance